# Week 6.3 — Arnoldi & Ritz Values on a Companion Matrix
Example on the companion matrix of `(z-1)^3`. Shows stagnation of GMRES for two steps, then finite termination at k=3.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import gmres
import pandas as pd

## Helper: vanilla Arnoldi (modified Gram-Schmidt)
Builds an orthonormal basis $V$ of the Krylov subspace $\mathcal{K}_m(A,v_1)=\text{span}\{v_1,Av_1,\dots,A^{m-1}v_1\}$ together with the upper Hessenberg matrix $H$ satisfying $AV_m=V_{m+1}H$ — the same decomposition underlying GMRES internally.

In [2]:
def simple_arnoldi(A, v1, m):
    n = len(v1)
    V = np.zeros((n, m + 1), dtype=np.complex128)
    H = np.zeros((m + 1, m), dtype=np.complex128)
    V[:, 0] = v1 / np.linalg.norm(v1)
    for j in range(m):
        w = A @ V[:, j]
        for i in range(j + 1):
            H[i, j] = np.dot(V[:, i].conj(), w)
            w = w - H[i, j] * V[:, i]
        H[j + 1, j] = np.linalg.norm(w)
        if H[j + 1, j] < 1e-14:
            V[:, j + 1] = np.zeros(n)
            break
        V[:, j + 1] = w / H[j + 1, j]
    return V, H

## Problem setup
$A$ is the companion matrix of $(z-1)^3$, so it has a single eigenvalue $1$ with algebraic multiplicity 3 but only a *single* eigenvector (a defective, non-diagonalizable matrix) — a deliberately hard case for Krylov methods, which typically rely on a rich eigenvector structure to make fast progress.

In [3]:
A = np.array([[0, 1, 0],
              [0, 0, 1],
              [1, -3, 3]], dtype=float)
b = np.array([1, 0, 0], dtype=float)
x0 = np.zeros(3)
tol = 1e-14
maxit = 3

## GMRES residuals
For this defective matrix, GMRES's residual stays essentially flat for the first two iterations (no useful spectral information yet) and then drops to exactly zero at iteration 3 — GMRES's guaranteed finite termination in at most $n=3$ steps for any nonsingular $n\times n$ matrix.

In [4]:
resvec = []
def callback_gmres(rk):
    resvec.append(np.linalg.norm(rk))

# Initial residual for plot
resvec.append(np.linalg.norm(b - A @ x0))
x, flag = gmres(A, b, x0=x0, rtol=tol, maxiter=maxit, callback=callback_gmres)
relres = resvec[-1] / resvec[0]
iter_num = len(resvec) - 1

plt.semilogy(np.arange(len(resvec)), resvec, 'o-')
plt.grid(True)
plt.xlabel('Iteration')
plt.ylabel('||r_k||_2')
plt.title('GMRES on companion matrix: flat, flat, then zero at k=3')
plt.xticks(np.arange(maxit + 1))
plt.show()

print(f'GMRES: flag={flag}, iters={iter_num}, relres={relres:.2e}')

GMRES: flag=0, iters=3, relres=0.00e+00


/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73689/2668359096.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Small Arnoldi to extract Ritz values and a posteriori residuals
The eigenvalues of the small $m\times m$ Hessenberg matrix $H_k$ are the **Ritz values** — Krylov-subspace approximations to $A$'s eigenvalues. Their a posteriori residual bound $|h_{k+1,k}|\,|y_m|$ (using the last subdiagonal entry of $H$ and the last component of each Ritz vector) can be computed without ever forming $A$'s true eigenvectors, and shows how well each Ritz value has been resolved after only 3 Arnoldi steps.

In [5]:
m = 3
v1 = b / np.linalg.norm(b)
V, H = simple_arnoldi(A, v1, m)
Hk = H[:m, :m]
hk1k = H[m, m - 1]

eigvals, Y = np.linalg.eig(Hk)
theta = eigvals  # Ritz values

ritz_res = np.abs(hk1k) * np.abs(Y[-1, :])

results_df = pd.DataFrame({
    'RitzValue': theta,
    'ResidualBound': ritz_res
})
print(results_df)

            RitzValue  ResidualBound
0  0.999992-0.000005j            0.0
1  1.000000+0.000009j            0.0
2  1.000008-0.000004j            0.0
